# Bat Orientation Calls – Klassifikation von Fledermausarten

**Ziel:** Aus einem Spektrogramm-Bild eines Ortungsrufs die Fledermausart erkennen.

**Pflicht laut Folie:** Decision Tree (+ verwandte Verfahren), FFNN, CNN, Ergebnisse validieren, bestes Modell darstellen.

**Ablauf:**
1. Vorbereitung (Colab, Bibliotheken, Einstellungen)
2. Labels laden
3. Bilder zuordnen
4. Klassenverteilung anschauen
5. Bilder vorbereiten
6. Train-Test-Split
7. Baseline
8. Decision Tree
9. Random Forest
10. Gradient Boosting
11. FFNN
12. CNN
13. Vergleich aller Modelle
14. Bestes Modell im Detail

Führt die Zellen **von oben nach unten** aus.

## 1. Vorbereitung

**Nur in Google Colab:** Die nächste Zelle verbindet Google Drive. Legt vorher in eurem Drive einen Ordner `bat_projekt` an, darin:
- den Ordner `bilder` mit allen PNG-Spektrogrammen
- die Datei `labels.csv`

Wenn ihr lokal (z. B. Jupyter auf dem Laptop) arbeitet, diese Zelle überspringen.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/bat_projekt

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

**Einstellungen:** Hier könnt ihr später Dinge ändern, z. B. eine größere Bildgröße ausprobieren.

In [ ]:
BILD_ORDNER  = "bilder"       # Ordner mit den PNG-Bildern
LABEL_DATEI  = "labels.csv"   # Tabelle mit filename, species
BREITE, HOEHE = 128, 64       # Größe der Bilder nach dem Verkleinern
MIN_PRO_ART  = 20             # Arten mit weniger Rufen werden entfernt
ZUFALL       = 42             # fester Zufallswert -> alle bekommen dieselben Ergebnisse

## 2. Labels laden

Die Tabelle sagt zu jedem Bild, welche Art ruft. In der Tabelle enden die Namen auf `.jpg`, die Bilder sind aber `.png`. Darum benutzen wir nur die **ID** (die Zahl vor dem Punkt). 3 Zeilen sind doppelt, die entfernen wir.

In [ ]:
df = pd.read_csv(LABEL_DATEI)
df["id"] = df["filename"].str.split(".").str[0]
print("Zeilen in der Tabelle:", len(df))

df = df.drop_duplicates(subset="id").reset_index(drop=True)
print("Nach Entfernen der Duplikate:", len(df))
df.head()

## 3. Bilder zuordnen

Wir suchen für jede ID die passende Bilddatei. Danach prüfen wir: Gibt es Labels ohne Bild oder Bilder ohne Label?

In [ ]:
dateien = {}
for datei in os.listdir(BILD_ORDNER):
    if datei.lower().endswith((".png", ".jpg")):
        id_ = os.path.splitext(datei)[0]
        dateien[id_] = os.path.join(BILD_ORDNER, datei)

df["pfad"] = df["id"].map(dateien)

print("Bilder im Ordner:      ", len(dateien))
print("Labels mit Bild:       ", df["pfad"].notna().sum())
print("Labels ohne Bild:      ", df["pfad"].isna().sum())
print("Bilder ohne Label:     ", len(set(dateien) - set(df["id"])))

df = df.dropna(subset=["pfad"]).reset_index(drop=True)

## 4. Klassenverteilung

Wie viele Rufe gibt es pro Art? Wenn einige Arten sehr selten sind, kann das Modell sie nicht lernen (**Class Imbalance**).

In [ ]:
anzahl = df["species"].value_counts()
print(anzahl)

anzahl.plot(kind="bar", color="steelblue")
plt.title("Anzahl Rufe pro Art")
plt.ylabel("Anzahl")
plt.tight_layout()
plt.show()

**Entscheidung:** Arten mit weniger als `MIN_PRO_ART` Rufen (Mausohr, Wasserfledermaus) werden entfernt. Mit 4–5 Beispielen kann kein Modell eine Art lernen. Diese Entscheidung müsst ihr im Bericht begründen (oder Frau Vogl nach mehr Daten fragen).

In [ ]:
seltene_arten = anzahl[anzahl < MIN_PRO_ART].index.tolist()
print("Entfernt:", seltene_arten)

df = df[~df["species"].isin(seltene_arten)].reset_index(drop=True)
print("Übrig:", len(df), "Rufe,", df["species"].nunique(), "Arten")

## 5. Bilder vorbereiten

Die PNGs sind Matplotlib-Grafiken mit **weißem Rand, Achsen und Zahlen**. Das stört das Modell. Wir machen:
1. **Zuschneiden:** Nur den farbigen Spektrogramm-Bereich behalten. Die Spektrogramm-Farben sind „bunt“ (lila/orange), Rand und Text sind weiß/grau/schwarz.
2. **Graustufen:** hell = laut.
3. **Verkleinern** auf `BREITE × HOEHE`.
4. **Skalieren** auf Werte 0 bis 1.

Zuerst schauen wir ein Originalbild an.

In [ ]:
beispiel = np.array(Image.open(df["pfad"][0]).convert("RGB"))
print("Originalgröße (Höhe, Breite, Farben):", beispiel.shape)

plt.imshow(beispiel)
plt.title("Original: " + df["species"][0])
plt.show()

Jetzt finden wir den Zuschnitt. Alle Bilder haben dasselbe Layout, also reicht es, ihn **einmal** am ersten Bild zu bestimmen.

In [ ]:
rgb = beispiel.astype(int)
saettigung = rgb.max(axis=2) - rgb.min(axis=2)   # wie "bunt" ist jedes Pixel?
bunt = saettigung > 30

zeilen  = np.where(bunt.mean(axis=1) > 0.3)[0]  # Zeilen, die überwiegend bunt sind
spalten = np.where(bunt.mean(axis=0) > 0.3)[0]  # Spalten, die überwiegend bunt sind

oben, unten   = zeilen[0],  zeilen[-1] + 1
links, rechts = spalten[0], spalten[-1] + 1
print("Zuschnitt: oben", oben, "unten", unten, "links", links, "rechts", rechts)

plt.imshow(beispiel[oben:unten, links:rechts])
plt.title("Zugeschnitten – prüfen: kein weißer Rand, keine Achsen mehr?")
plt.show()

**Bitte prüfen:** Ist im zugeschnittenen Bild nur noch das Spektrogramm zu sehen? Falls nicht, die vier Zahlen oben von Hand anpassen, z. B. `oben, unten, links, rechts = 20, 380, 60, 720`.

Jetzt laden wir **alle** Bilder auf diese Weise.

In [ ]:
bilder = []
for pfad in df["pfad"]:
    bild = Image.open(pfad).convert("RGB")
    bild = bild.crop((links, oben, rechts, unten))   # Rand und Achsen weg
    bild = bild.convert("L")                         # Graustufen
    bild = bild.resize((BREITE, HOEHE))              # gleiche Größe
    arr = np.asarray(bild, dtype=np.float32) / 255.0 # Werte 0..1
    bilder.append(arr)

bilder = np.stack(bilder)
arten = df["species"].values
print("Bilder:", bilder.shape, "  (Anzahl, Höhe, Breite)")

**Beispielbilder pro Art** (gut für den Bericht): Kann man Unterschiede zwischen den Arten sehen?

In [ ]:
art_namen = sorted(set(arten))
pro_art = 3
fig, ax = plt.subplots(len(art_namen), pro_art, figsize=(3 * pro_art, 2 * len(art_namen)))

for i in range(len(art_namen)):
    indizes = np.where(arten == art_namen[i])[0][:pro_art]
    for j in range(len(indizes)):
        ax[i, j].imshow(bilder[indizes[j]], aspect="auto", cmap="magma")
        ax[i, j].set_title(art_namen[i], fontsize=9)
        ax[i, j].axis("off")

plt.tight_layout()
plt.show()

## 6. Train-Test-Split

- Die Artnamen werden in Zahlen umgewandelt (0, 1, 2, 3).
- 80 % Training, 20 % Test.
- `stratify=y`: Jede Art ist im Test im gleichen Verhältnis wie im Training.
- **Das Testset benutzen wir erst ganz am Ende!** Einstellungen suchen wir mit Cross-Validation auf dem Training.

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(arten)
print("Klassen:", list(encoder.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    bilder, y, test_size=0.2, stratify=y, random_state=ZUFALL)

print("Training:", X_train.shape[0], "Bilder")
print("Test:    ", X_test.shape[0], "Bilder")

**Bilder flach machen:** Decision Tree, Random Forest, Boosting und FFNN können kein Bild lesen, nur eine Zeile mit Zahlen. Darum wird jedes Bild (64 × 128) zu einer Zeile mit 8192 Zahlen. Jedes Pixel ist dann ein Feature.

In [ ]:
X_train_flach = X_train.reshape(len(X_train), -1)
X_test_flach  = X_test.reshape(len(X_test), -1)
print("Flach:", X_train_flach.shape)

# 5-fache Cross-Validation (für alle Modelle gleich)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=ZUFALL)

# hier sammeln wir die Vorhersagen aller Modelle auf dem Testset
vorhersagen = {}

## 7. Baseline

Ein „dummes“ Modell, das **immer die häufigste Art** sagt. Jedes echte Modell muss deutlich besser sein.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_flach, y_train)
vorhersagen["Baseline"] = baseline.predict(X_test_flach)

print("Baseline Accuracy:", round(accuracy_score(y_test, vorhersagen["Baseline"]), 3))

## 8. Decision Tree

Ein Baum stellt Ja/Nein-Fragen an die Pixel, z. B. „Ist Pixel Nr. 1532 heller als 0,6?“.

Ein zu tiefer Baum lernt die Trainingsbilder auswendig (**Overfitting**). Darum suchen wir die beste Tiefe mit **Cross-Validation**. `class_weight="balanced"` sorgt dafür, dass kleine Arten nicht untergehen.

In [ ]:
suche = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=ZUFALL),
    param_grid={"max_depth": [2, 3, 5, 8, 12, 20, None]},
    cv=cv, scoring="f1_macro", return_train_score=True)
suche.fit(X_train_flach, y_train)

print("Beste Tiefe:", suche.best_params_["max_depth"])
print("CV Macro-F1:", round(suche.best_score_, 3))
baum = suche.best_estimator_
vorhersagen["Decision Tree"] = baum.predict(X_test_flach)

**Diagramm für den Bericht:** Training- und Validierungs-Score je nach Baumtiefe. Wo die Kurven auseinanderlaufen, beginnt Overfitting.

In [ ]:
ergebnisse = pd.DataFrame(suche.cv_results_)
tiefen = [str(t) for t in ergebnisse["param_max_depth"]]

plt.plot(tiefen, ergebnisse["mean_train_score"], marker="o", label="Training")
plt.plot(tiefen, ergebnisse["mean_test_score"], marker="o", label="Validierung (CV)")
plt.xlabel("max_depth")
plt.ylabel("Macro-F1")
plt.title("Decision Tree: Tiefe vs. Leistung")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(16, 6))
plot_tree(baum, max_depth=2, class_names=list(encoder.classes_), filled=True, fontsize=8)
plt.title("Die ersten Ebenen des Baums")
plt.show()

## 9. Random Forest (verwandtes Verfahren)

Viele Bäume, jeder mit einem zufälligen Teil der Daten und Pixel. Bei der Vorhersage **stimmen alle ab**. Das ist stabiler als ein einzelner Baum (weniger Varianz).

In [ ]:
wald = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                              n_jobs=-1, random_state=ZUFALL)

scores = cross_val_score(wald, X_train_flach, y_train, cv=cv, scoring="f1_macro")
print("CV Macro-F1:", round(scores.mean(), 3), "+/-", round(scores.std(), 3))

wald.fit(X_train_flach, y_train)
vorhersagen["Random Forest"] = wald.predict(X_test_flach)

**Feature Importance als Bild:** Welche Bereiche des Spektrogramms (welche Frequenzen, welche Zeitpunkte) benutzt der Wald am meisten?

In [ ]:
wichtigkeit = wald.feature_importances_.reshape(HOEHE, BREITE)

plt.imshow(wichtigkeit, aspect="auto", cmap="hot")
plt.colorbar(label="Wichtigkeit")
plt.title("Random Forest: wichtige Pixel")
plt.xlabel("Zeit")
plt.ylabel("Frequenz (oben = hoch)")
plt.show()

## 10. Gradient Boosting (verwandtes Verfahren)

Bäume werden **nacheinander** gebaut. Jeder neue Baum korrigiert die Fehler der vorherigen. Das kann ein paar Minuten dauern.

In [ ]:
boost = HistGradientBoostingClassifier(class_weight="balanced", random_state=ZUFALL)

scores = cross_val_score(boost, X_train_flach, y_train, cv=cv, scoring="f1_macro")
print("CV Macro-F1:", round(scores.mean(), 3), "+/-", round(scores.std(), 3))

boost.fit(X_train_flach, y_train)
vorhersagen["Gradient Boosting"] = boost.predict(X_test_flach)

## 11. FFNN (Feed-Forward Neural Network)

Ein neuronales Netz mit 2 versteckten Schichten (256 und 64 Neuronen). Eingabe: die 8192 flachen Pixel.

`early_stopping=True` stoppt das Training, wenn es auf einem Validierungsteil nicht mehr besser wird (gegen Overfitting).

**Schwäche:** Das FFNN weiß nicht, welche Pixel nebeneinander liegen. Ein leicht verschobener Ruf sieht für das FFNN ganz anders aus.

In [ ]:
ffnn = MLPClassifier(hidden_layer_sizes=(256, 64), alpha=1e-3,
                     early_stopping=True, max_iter=300, random_state=ZUFALL)

scores = cross_val_score(ffnn, X_train_flach, y_train, cv=cv, scoring="f1_macro")
print("CV Macro-F1:", round(scores.mean(), 3), "+/-", round(scores.std(), 3))

ffnn.fit(X_train_flach, y_train)
vorhersagen["FFNN"] = ffnn.predict(X_test_flach)

plt.plot(ffnn.loss_curve_)
plt.title("FFNN: Loss pro Epoche")
plt.xlabel("Epoche"); plt.ylabel("Loss")
plt.show()

## 12. CNN (Convolutional Neural Network)

Das CNN bekommt das Bild **als Bild**, nicht flach.
- **Conv2D** sucht kleine Muster (Kanten, schräge Linien) überall im Bild.
- **MaxPooling2D** verkleinert das Bild Schritt für Schritt. So erkennen spätere Schichten größere Formen.
- **RandomTranslation** verschiebt die Bilder beim Training zufällig etwas in der Zeitrichtung (Data Augmentation). So gibt es künstlich mehr Daten.
- **Dropout** schaltet zufällig Neuronen aus (gegen Overfitting).
- **EarlyStopping** stoppt, wenn die Validierung nicht mehr besser wird.

**In Colab:** Laufzeit → Laufzeittyp ändern → **GPU**, sonst dauert es lange.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(ZUFALL)
anzahl_klassen = len(encoder.classes_)

cnn = tf.keras.Sequential([
    layers.Input(shape=(HOEHE, BREITE, 1)),
    layers.RandomTranslation(0, 0.1),
    layers.Conv2D(16, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(anzahl_klassen, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

In [ ]:
gewichte = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
klassen_gewichte = dict(enumerate(gewichte))

verlauf = cnn.fit(
    X_train[..., np.newaxis], y_train,      # [..., np.newaxis]: Bild bekommt 1 Farbkanal
    validation_split=0.15,
    epochs=40, batch_size=32,
    class_weight=klassen_gewichte,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

**Lernkurve für den Bericht:** Laufen Training und Validierung stark auseinander, ist das Overfitting.

In [ ]:
plt.plot(verlauf.history["accuracy"], label="Training")
plt.plot(verlauf.history["val_accuracy"], label="Validierung")
plt.xlabel("Epoche"); plt.ylabel("Accuracy")
plt.title("CNN Lernkurve")
plt.legend()
plt.show()

wahrscheinlichkeiten = cnn.predict(X_test[..., np.newaxis], verbose=0)
vorhersagen["CNN"] = np.argmax(wahrscheinlichkeiten, axis=1)

## 13. Vergleich aller Modelle (auf dem Testset)

- **Accuracy:** Anteil richtig erkannter Rufe.
- **Macro-F1:** F1-Wert pro Art, dann gemittelt. Jede Art zählt gleich viel, auch die kleinen. Wegen der ungleichen Klassen ist das **fairer als Accuracy**. Das beste Modell wählen wir nach Macro-F1.

In [ ]:
zeilen = []
for name in vorhersagen:
    p = vorhersagen[name]
    zeilen.append({"Modell": name,
                   "Accuracy": round(accuracy_score(y_test, p), 3),
                   "Macro-F1": round(f1_score(y_test, p, average="macro"), 3)})

vergleich = pd.DataFrame(zeilen).sort_values("Macro-F1", ascending=False).reset_index(drop=True)
vergleich

In [ ]:
vergleich.plot(x="Modell", y=["Accuracy", "Macro-F1"], kind="bar", figsize=(9, 4))
plt.title("Modellvergleich auf dem Testset")
plt.ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 14. Bestes Modell im Detail

- **Classification Report:** Precision, Recall und F1 für jede Art.
- **Confusion Matrix:** Zeilen = echte Art, Spalten = vorhergesagte Art. Auf der Diagonale stehen die richtigen Vorhersagen, daneben die Verwechslungen.

In [ ]:
bestes = vergleich["Modell"][0]
p_bestes = vorhersagen[bestes]
print("Bestes Modell:", bestes, "\n")

print(classification_report(y_test, p_bestes, target_names=encoder.classes_, zero_division=0))

ConfusionMatrixDisplay(confusion_matrix(y_test, p_bestes),
                       display_labels=encoder.classes_).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix – " + bestes)
plt.tight_layout()
plt.show()

**Falsch erkannte Rufe anschauen:** Warum könnte sich das Modell geirrt haben? Sehen die Rufe der verwechselten Arten ähnlich aus?

In [ ]:
falsch = np.where(p_bestes != y_test)[0]
print("Falsch erkannt:", len(falsch), "von", len(y_test))

anzeigen = falsch[:6]
if len(anzeigen) > 0:
    fig, ax = plt.subplots(1, len(anzeigen), figsize=(3 * len(anzeigen), 2.5), squeeze=False)
    for j in range(len(anzeigen)):
        k = anzeigen[j]
        ax[0, j].imshow(X_test[k], aspect="auto", cmap="magma")
        ax[0, j].set_title("echt: " + encoder.classes_[y_test[k]][:10] +
                           "\nModell: " + encoder.classes_[p_bestes[k]][:10], fontsize=8)
        ax[0, j].axis("off")
    plt.tight_layout()
    plt.show()

## Was in den Bericht gehört

1. Klassenverteilung (Abschnitt 4) und eure Entscheidung zu den seltenen Arten
2. Beispielbilder pro Art (Abschnitt 5)
3. Diagramm Baumtiefe (Abschnitt 8) und Feature Importance (Abschnitt 9)
4. CNN Lernkurve (Abschnitt 12)
5. Vergleichstabelle und Diagramm (Abschnitt 13)
6. Confusion Matrix und Report des besten Modells (Abschnitt 14)
7. Diskussion: Welche Arten werden verwechselt? Grenzen (wenige Daten, Imbalance)?

**Ideen zum Verbessern:** größere Bilder (z. B. `BREITE, HOEHE = 256, 128`), mehr Daten von Frau Vogl, mehr Conv-Schichten im CNN.